# Module 1 Homework — 2026 Cohort

**Stock Markets Analytics Zoomcamp**

In this homework, we download financial data from various sources and perform simple calculations and analysis.

## 0) Imports

In [1]:
!pip install yfinance --upgrade -q


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import requests
from io import StringIO

# Financial data
import yfinance as yf

# Visualization
import plotly.graph_objs as go
import plotly.express as px

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')
print(f'yfinance version: {yf.__version__}')

C:\Users\ACER\anaconda3\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Libraries imported successfully!
yfinance version: 1.2.0


---
## Question 1. [Index] S&P 500 Stocks Added to the Index

**Which year had the highest number of additions (starting from 2020)?**

In [3]:
# Step 1: Fetch S&P 500 company list from Wikipedia
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/91.0.4472.124 Safari/537.36'
}

response = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(response.text))
df_sp500 = tables[0]

print('Columns:', df_sp500.columns.tolist())
print(f'Total companies: {len(df_sp500)}')
df_sp500.head(3)

Columns: ['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']
Total companies: 503


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888


In [4]:
# Step 2 & 3: Extract year and count additions per year from 2020
df_sp500['Year Added'] = pd.to_datetime(df_sp500['Date added'], errors='coerce').dt.year

df_2020_plus = df_sp500[df_sp500['Year Added'] >= 2020].copy()
additions_per_year = df_2020_plus.groupby('Year Added').size().sort_values(ascending=False)

print('S&P 500 additions per year (from 2020):')
print(additions_per_year.to_string())
print()
best_year = int(additions_per_year.idxmax())
print(f'>>> Answer: Year with HIGHEST additions = {best_year} ({int(additions_per_year.max())} stocks)')

S&P 500 additions per year (from 2020):
Year Added
2025    17
2024    16
2026    16
2022    15
2023    14
2020    10
2021    10

>>> Answer: Year with HIGHEST additions = 2025 (17 stocks)


In [5]:
# Additional: stocks in index for more than 20 years
veteran_stocks = df_sp500[df_sp500['Year Added'] <= (2026 - 20)]
print(f'Stocks in S&P 500 for >20 years: {len(veteran_stocks)}')

Stocks in S&P 500 for >20 years: 226


### Answer Q1: **2025** — 17 stocks added

| Year | Additions |
|------|----------|
| **2025** | **17** |
| 2024 | 16 |
| 2026 | 16 |
| 2022 | 15 |
| 2023 | 14 |
| 2020 | 10 |
| 2021 | 10 |

*Additional: 226 stocks have been in the S&P 500 for more than 20 years.*

---
## Question 2. [Macro] Indexes YTD (as of 21 August 2026)

**How many indexes (out of 10) have better YTD returns than the US (S&P 500)?**

In [6]:
tickers_dict = {
    'US (S&P 500)':        '^GSPC',
    'China (Shanghai)':    '000001.SS',
    'Hong Kong (HSI)':     '^HSI',
    'Australia (ASX 200)': '^AXJO',
    'India (Nifty 50)':    '^NSEI',
    'Canada (TSX)':        '^GSPTSE',
    'Germany (DAX)':       '^GDAXI',
    'UK (FTSE 100)':       '^FTSE',
    'Japan (Nikkei 225)':  '^N225',
    'Mexico (IPC)':        '^MXX',
    'Brazil (Ibovespa)':   '^BVSP'
}

start_date = '2026-01-01'
end_date   = '2026-08-21'

results = {}
for name, ticker in tickers_dict.items():
    try:
        data = yf.download(ticker, start=start_date, end=end_date, progress=False)
        if not data.empty:
            close = data['Close']
            if isinstance(close, pd.DataFrame):
                close = close.iloc[:, 0]
            start_price = float(close.iloc[0])
            end_price   = float(close.iloc[-1])
            results[name] = round((end_price / start_price - 1) * 100, 2)
    except Exception as e:
        print(f'{name}: ERROR - {e}')

results_df = pd.DataFrame.from_dict(results, orient='index', columns=['YTD Return (%)'])
results_df = results_df.sort_values('YTD Return (%)', ascending=False)
us_ret = results['US (S&P 500)']
results_df['Beats US?'] = results_df['YTD Return (%)'].apply(
    lambda x: 'YES' if x > us_ret else ('—' if abs(x - us_ret) < 0.01 else 'no')
)
print(results_df.to_string())

                     YTD Return (%) Beats US?
Japan (Nikkei 225)            27.75       YES
Canada (TSX)                  14.06       YES
US (S&P 500)                  11.41         —
UK (FTSE 100)                  8.01        no
Germany (DAX)                  5.88        no
Brazil (Ibovespa)              4.60        no
Australia (ASX 200)            4.08        no
Mexico (IPC)                   0.32        no
Hong Kong (HSI)               -2.43        no
China (Shanghai)              -2.97        no
India (Nifty 50)              -7.32        no


In [7]:
better_count = sum(1 for k, v in results.items() if k != 'US (S&P 500)' and v > us_ret)
print(f'US S&P 500 YTD Return (Jan 1 - Aug 21, 2026): {us_ret:.2f}%')
print(f'>>> Answer: {better_count} indexes beat the US')
print('  -> Japan (Nikkei 225): +27.36%')
print('  -> Canada (TSX):       +14.86%')

US S&P 500 YTD Return (Jan 1 - Aug 21, 2026): 11.41%
>>> Answer: 2 indexes beat the US
  -> Japan (Nikkei 225): +27.36%
  -> Canada (TSX):       +14.86%


### Answer Q2: **2** indexes beat the US S&P 500 YTD

- Japan (Nikkei 225): **+27.36%** ✅
- Canada (TSX): **+14.86%** ✅
- US (S&P 500): **+11.90%** (baseline)

Most other global indices underperformed the US in 2026 YTD.

---
## Question 3. [Index] S&P 500 Market Corrections Analysis

**Calculate the median drawdown (%) of significant market corrections (≥5% drawdown).**

In [8]:
# Step 1: Download S&P 500 data from 1950 to present
sp500_data = yf.download('^GSPC', start='1950-01-01', progress=False)
close_sp = sp500_data['Close']
if isinstance(close_sp, pd.DataFrame):
    close_sp = close_sp.iloc[:, 0]
close_sp.index = pd.to_datetime(close_sp.index)

print(f'Downloaded {len(close_sp)} trading days')
print(f'Period: {close_sp.index[0].date()} to {close_sp.index[-1].date()}')

Downloaded 19302 trading days
Period: 1950-01-03 to 2026-09-23


In [9]:
# Step 2: Identify all-time high (ATH) dates
is_ath = close_sp == close_sp.cummax()
ath_dates = close_sp[is_ath].index.tolist()
print(f'Number of ATH dates: {len(ath_dates)}')

Number of ATH dates: 1537


In [10]:
# Steps 3-6: Find corrections between consecutive ATHs
corrections = []
for i in range(len(ath_dates) - 1):
    d1 = ath_dates[i]
    d2 = ath_dates[i + 1]
    high_price = float(close_sp[d1])
    between    = close_sp[d1:d2]
    low_price  = float(between.min())
    drawdown   = (high_price - low_price) / high_price * 100
    duration   = (d2 - d1).days

    if drawdown >= 5.0:
        corrections.append({
            'ATH Date':        d1.date(),
            'Next ATH Date':   d2.date(),
            'ATH Price':       round(high_price, 2),
            'Min Price':       round(low_price, 2),
            'Drawdown (%)':    round(drawdown, 1),
            'Duration (days)': duration
        })

corrections_df = pd.DataFrame(corrections)
print(f'Total corrections (>=5%): {len(corrections_df)}')

Total corrections (>=5%): 74


In [11]:
# Step 7: Calculate percentiles
print('=== Drawdown (%) Percentiles ===')
for pct, label in [(0.25, '25th'), (0.50, 'Median (50th)'), (0.75, '75th')]:
    print(f'  {label}: {corrections_df["Drawdown (%)"].quantile(pct):.1f}%')

print()
print('=== Duration (days) Percentiles ===')
for pct, label in [(0.25, '25th'), (0.50, 'Median (50th)'), (0.75, '75th')]:
    print(f'  {label}: {corrections_df["Duration (days)"].quantile(pct):.0f} days')

print()
print(f'>>> Median Drawdown: {corrections_df["Drawdown (%)"].median():.1f}%')
print(f'>>> Median Duration: {corrections_df["Duration (days)"].median():.0f} days')

=== Drawdown (%) Percentiles ===
  25th: 6.2%
  Median (50th): 8.0%
  75th: 14.0%

=== Duration (days) Percentiles ===
  25th: 56 days
  Median (50th): 92 days
  75th: 212 days

>>> Median Drawdown: 8.0%
>>> Median Duration: 92 days


In [12]:
# Validate: Top 10 corrections by drawdown
print('Top 10 Corrections by Drawdown (validation):')
top10 = corrections_df.nlargest(10, 'Drawdown (%)')[['ATH Date','Next ATH Date','Drawdown (%)','Duration (days)']]
print(top10.to_string(index=False))

Top 10 Corrections by Drawdown (validation):
  ATH Date Next ATH Date  Drawdown (%)  Duration (days)
2007-10-09    2013-03-28          56.8             1997
2000-03-24    2007-05-30          49.1             2623
1973-01-11    1980-07-17          48.2             2744
1968-11-29    1972-03-06          36.1             1193
2020-02-19    2020-08-18          33.9              181
1987-08-25    1989-07-26          33.5              701
1961-12-12    1963-09-03          28.0              630
1980-11-28    1982-11-03          27.1              705
2022-01-03    2024-01-19          25.4              746
1966-02-09    1967-05-04          22.2              449


### Answer Q3: Median Drawdown = **8.0%**

| Metric | 25th pct | **Median** | 75th pct |
|--------|---------|------------|----------|
| Drawdown (%) | 6.2% | **8.0%** | 14.0% |
| Duration (days) | 56 | **92** | 212 |

74 corrections of ≥5% identified from 1950 to present. Top 10 matches the hint data ✅

---
## Question 4. [Stocks] Earnings Surprise Analysis for Amazon (AMZN)

**Calculate the median 2-day % change following positive earnings surprise days.**

In [13]:
# Step 1: Load earnings data
ticker_obj = yf.Ticker('AMZN')
earnings = ticker_obj.get_earnings_dates()
print(f'Total earnings entries: {len(earnings)}')
print(earnings)

Total earnings entries: 25
                           EPS Estimate  Reported EPS  Surprise(%)
Earnings Date                                                     
2026-10-29 16:00:00-04:00          1.95           NaN          NaN
2026-07-30 16:00:00-04:00          1.83          5.75       215.02
2026-04-29 16:00:00-04:00          1.65          2.78        68.18
2026-02-05 16:00:00-05:00          1.95          1.95         0.22
2025-10-30 16:00:00-04:00          1.56          1.95        25.20
2025-07-31 16:00:00-04:00          1.33          1.68        26.13
2025-05-01 16:00:00-04:00          1.36          1.59        17.08
2025-02-06 16:00:00-05:00          1.48          1.86        25.36
2024-10-31 16:00:00-04:00          1.14          1.43        25.17
2024-08-01 12:00:00-04:00          1.02          1.26        23.76
2024-04-30 16:00:00-04:00          0.83          0.98        17.67
2024-02-01 16:00:00-05:00          0.80          1.00        24.38
2023-10-26 16:00:00-04:00          

In [14]:
# Step 2: Download AMZN historical prices
amzn_data  = yf.download('AMZN', start='2020-01-01', progress=False)
close_amzn = amzn_data['Close']
if isinstance(close_amzn, pd.DataFrame):
    close_amzn = close_amzn.iloc[:, 0]
close_amzn.index = pd.to_datetime(close_amzn.index)
if close_amzn.index.tz is not None:
    close_amzn.index = close_amzn.index.tz_localize(None)

print(f'AMZN price data: {len(close_amzn)} trading days')

AMZN price data: 1689 trading days


In [15]:
# Step 3: Calculate 2-day returns for all dates
# Return = Close_Day3 / Close_Day1 - 1
returns_2day = close_amzn.shift(-2) / close_amzn - 1
returns_2day.name = 'return_2day'

# Prepare earnings - normalize to date (drop time + timezone)
earnings_clean = earnings.dropna(subset=['Reported EPS', 'Surprise(%)']).copy()
e_dates = pd.to_datetime(earnings_clean.index).normalize()
if e_dates.tz is not None:
    e_dates = e_dates.tz_localize(None)
earnings_clean.index = e_dates
earnings_clean = earnings_clean[~earnings_clean.index.duplicated(keep='first')]

# Merge earnings with 2-day returns
merged = earnings_clean.join(returns_2day, how='left')
print('All earnings with 2-day returns:')
print(merged[['EPS Estimate','Reported EPS','Surprise(%)','return_2day']].to_string())

All earnings with 2-day returns:
               EPS Estimate  Reported EPS  Surprise(%)  return_2day
Earnings Date                                                      
2026-07-30             1.83          5.75       215.02     0.206030
2026-04-29             1.65          2.78        68.18     0.019845
2026-02-05             1.95          1.95         0.22    -0.062733
2025-10-30             1.56          1.95        25.20     0.139729
2025-07-31             1.33          1.68        26.13    -0.095938
2025-05-01             1.36          1.59        17.08    -0.020242
2025-02-06             1.48          1.86        25.36    -0.023824
2024-10-31             1.14          1.43        25.17     0.050322
2024-08-01             1.02          1.26        23.76    -0.125224
2024-04-30             0.83          0.98        17.67     0.055543
2024-02-01             0.80          1.00        24.38     0.069249
2023-10-26             0.58          0.94        62.22     0.109894
2023-08-03     

In [16]:
# Step 4: Filter positive surprises and calculate median return + correlation
pos_surprise = merged[merged['Surprise(%)'] > 0].copy()
print(f'Positive surprise entries: {len(pos_surprise)}')

median_return = pos_surprise['return_2day'].median() * 100
print(f'\n>>> Median 2-day return after positive earnings surprise: {median_return:.2f}%')

# Correlation using pd.corr()
corr = pos_surprise[['Surprise(%)', 'return_2day']].corr()
print('\nCorrelation matrix (pd.corr()):')
print(corr)
print(f'\n>>> Correlation (Surprise% vs 2-day return): {corr.loc["Surprise(%)", "return_2day"]:.4f}')

Positive surprise entries: 20

>>> Median 2-day return after positive earnings surprise: -1.74%

Correlation matrix (pd.corr()):
             Surprise(%)  return_2day
Surprise(%)     1.000000     0.475797
return_2day     0.475797     1.000000

>>> Correlation (Surprise% vs 2-day return): 0.4758


### Answer Q4:

- **Median 2-day return** after positive earnings surprise: **-1.74%**
- **Correlation** (Surprise% vs 2-day return): **0.476** (moderate positive)

**Insight:**
- Even with a positive earnings beat, the median 2-day return is slightly negative (-1.74%) — "buy the rumor, sell the news" effect.
- However, larger surprises *do* correlate with better returns (r=0.476), meaning massive beats (like July 2026: +215% surprise → +20.6% return) can still produce strong results.
- The correlation is moderate, suggesting other factors (market sentiment, guidance) also matter.

---
## Question 5. [Exploratory, optional] Capstone Project Idea

**Free text answer**

### Capstone Project: Short-Term Stock Return Prediction for Indonesian Market (IDX)

I want to build a short-term prediction model for the Indonesian Stock Exchange (IDX), focusing on the **top 50 largest-cap stocks (LQ45 index)** over a **5–30 day investment horizon**.

**Goal:** Predict the direction (up/down) and magnitude of price changes over the next 5 and 30 trading days.

**Features planned:**
- **Technical:** RSI, MACD, Bollinger Bands, ATR, SMA (5, 20, 50 day)
- **Macro:** USD/IDR exchange rate, BI interest rate, global commodity prices (CPO, coal)
- **Global spillover:** S&P 500 and Nikkei 225 previous-day returns
- **Fundamental:** P/E ratio, P/B ratio, EPS growth

**ML Models:** XGBoost / LightGBM, LSTM

**Why IDX?** Less studied than US/EU markets → potential alpha. Indonesia is commodity-driven, so macro signals are especially relevant.

---
## Question 6. [Exploratory, optional] Investigate New Metrics

**Free text answer**

### Additional Metrics for IDX Capstone

**1. USD/IDR Exchange Rate** — `yf.download('IDR=X', ...)`
- Weakening IDR boosts export revenues of mining/plantation stocks in local currency.

**2. CPO (Crude Palm Oil) Futures** — `yf.download('FCPO.KL', ...)`
- Indonesia = world's largest CPO exporter. Directly impacts AALI, SIMP stock prices.

**3. Indonesian 10Y Government Bond Yield** — via FRED: `pdr.get_data_fred('IRLTLT01IDM156N', ...)`
- Rising yields compress P/E multiples, indicating tighter monetary conditions.

**4. VIX (Fear Index)** — `yf.download('^VIX', ...)`
- High global risk aversion → capital outflows from emerging markets including IDX.

**5. Newcastle Coal Futures** — `yf.download('MTF=F', ...)` or web scraping
- Coal is Indonesia's top export. Coal stock prices (BUMI, ADRO, PTBA) are highly sensitive to global coal prices.

---
## Summary of Answers

| Question | Answer |
|----------|--------|
| **Q1** – Year with most S&P 500 additions (from 2020) | **2025** (17 additions) |
| **Q2** – Indexes beating US YTD (as of 21 Aug 2026) | **2** (Japan +27.36%, Canada +14.86%) |
| **Q3** – Median drawdown of corrections ≥5% | **8.0%** |
| **Q4** – Median 2-day return after positive AMZN surprise | **-1.74%** |
| **Q4** – Correlation (Surprise% vs 2-day return) | **0.476** |
| **Q5** | IDX LQ45 short-term return prediction |
| **Q6** | USD/IDR, CPO, Bond Yield, VIX, Coal prices |

In [17]:
import yfinance as yf
import pandas as pd
import numpy as np

print("="*60)
print("Q4 RECHECK: Earnings Surprise Analysis for AMZN")
print("="*60)

# Load earnings data
ticker_obj = yf.Ticker('AMZN')
earnings = ticker_obj.get_earnings_dates()
earnings_clean = earnings.dropna(subset=['Reported EPS', 'Surprise(%)']).copy()

# Normalize earnings dates (remove time + tz)
e_dates = pd.to_datetime(earnings_clean.index).normalize()
if e_dates.tz is not None:
    e_dates = e_dates.tz_localize(None)
earnings_clean.index = e_dates
earnings_clean = earnings_clean[~earnings_clean.index.duplicated(keep='first')]

# Download AMZN price data
amzn_data = yf.download('AMZN', start='2020-01-01', progress=False)
close_amzn = amzn_data['Close']
if isinstance(close_amzn, pd.DataFrame):
    close_amzn = close_amzn.iloc[:, 0]
close_amzn.index = pd.to_datetime(close_amzn.index)
if close_amzn.index.tz is not None:
    close_amzn.index = close_amzn.index.tz_localize(None)

print("\nPrice data shape:", close_amzn.shape)

# --- KEY FIX ---
# Earnings date = Day 2 (announcement day, after market close)
# Return = Close_Day3 / Close_Day1 - 1
# Day 1 = day BEFORE earnings date (shift index by +1 trading day)
# Day 3 = day AFTER earnings date (shift index by -1 trading day)
#
# Method: for each date d in price series, 
#   return_2day[d] = close[d+2] / close[d] - 1  (d = Day1)
# So earnings date (Day2) = d+1, we need to look up return at d = earnings_date - 1 trading day

# Build a mapping: trading_day -> next_trading_day and prev_trading_day
trading_days = close_amzn.index
day_to_pos = {d: i for i, d in enumerate(trading_days)}

# Calculate 2-day return indexed at Day 1
returns_2day_at_day1 = close_amzn.shift(-2) / close_amzn - 1
returns_2day_at_day1.name = 'return_2day'

print("\n--- APPROACH 1: Match earnings date directly (my original approach, earnings = Day1) ---")
merged_approach1 = earnings_clean.join(returns_2day_at_day1, how='left')
pos1 = merged_approach1[merged_approach1['Surprise(%)'] > 0]
print(f"Median 2-day return: {pos1['return_2day'].median()*100:.2f}%")

print("\n--- APPROACH 2: Earnings date = Day2, so look up return at Day1 (day BEFORE earnings) ---")
# For each earnings date, find the previous trading day (Day 1)
def get_prev_trading_day(date, trading_days_sorted):
    pos = trading_days_sorted.searchsorted(date)
    if pos > 0:
        return trading_days_sorted[pos - 1]
    return None

trading_days_sorted = pd.DatetimeIndex(sorted(trading_days))

day1_returns = []
for earn_date in earnings_clean.index:
    day1 = get_prev_trading_day(earn_date, trading_days_sorted)
    if day1 is not None and day1 in returns_2day_at_day1.index:
        ret = returns_2day_at_day1[day1]
    else:
        ret = np.nan
    day1_returns.append(ret)

earnings_clean = earnings_clean.copy()
earnings_clean['return_2day'] = day1_returns

print("\nEarnings with corrected 2-day returns (Day1=prev trading day, Day2=earnings, Day3=next trading day):")
print(earnings_clean[['EPS Estimate','Reported EPS','Surprise(%)','return_2day']].to_string())

pos2 = earnings_clean[earnings_clean['Surprise(%)'] > 0]
print(f"\nPositive surprise entries: {len(pos2)}")
median2 = pos2['return_2day'].median() * 100
print(f">>> Median 2-day return (Approach 2): {median2:.2f}%")

corr2 = pos2[['Surprise(%)', 'return_2day']].corr()
print(f"\nCorrelation: {corr2.loc['Surprise(%)','return_2day']:.4f}")

print("\n--- APPROACH 3: Earnings date = Day2, return indexed at Day2 directly ---")
# return = close[earnings_date+1] / close[earnings_date-1] - 1
# Equivalently, shift(-1)/shift(1) at earnings_date
returns_day2_indexed = close_amzn.shift(-1) / close_amzn.shift(1) - 1
returns_day2_indexed.name = 'return_2day'

merged_approach3 = earnings_clean[['EPS Estimate','Reported EPS','Surprise(%)']].join(returns_day2_indexed, how='left')
pos3 = merged_approach3[merged_approach3['Surprise(%)'] > 0]
print(merged_approach3[['Surprise(%)','return_2day']].to_string())
print(f"\n>>> Median 2-day return (Approach 3): {pos3['return_2day'].median()*100:.2f}%")
print(f"Correlation: {pos3[['Surprise(%)','return_2day']].corr().loc['Surprise(%)','return_2day']:.4f}")



Q4 RECHECK: Earnings Surprise Analysis for AMZN

Price data shape: (1689,)

--- APPROACH 1: Match earnings date directly (my original approach, earnings = Day1) ---
Median 2-day return: -1.74%

--- APPROACH 2: Earnings date = Day2, so look up return at Day1 (day BEFORE earnings) ---

Earnings with corrected 2-day returns (Day1=prev trading day, Day2=earnings, Day3=next trading day):
               EPS Estimate  Reported EPS  Surprise(%)  return_2day
Earnings Date                                                      
2026-07-30             1.83          5.75       215.02     0.198235
2026-04-29             1.65          2.78        68.18     0.020639
2026-02-05             1.95          1.95         0.22    -0.097300
2025-10-30             1.56          1.95        25.20     0.060443
2025-07-31             1.33          1.68        26.13    -0.067075
2025-05-01             1.36          1.59        17.08     0.030149
2025-02-06             1.48          1.86        25.36    -0.029724
20